# Module 1: Build a deep agent

> ~45 min · optional

You build the client-research agent one piece at a time with `deepagents`: built-in tools, your own tools, planning, subagents, memory, middleware, approvals, and skills. Every run is a trace in LangSmith, so you see what each piece adds. Two beta sections let the agent write code that drives its own tools and subagents. Modules 2–5 run a packaged version of this agent.

<img src="../images/deepAgentsDiag.png" style="width: auto; max-height: 420px; border-radius: 8px;">

## Setup
This cell loads your keys and the model and points tracing at the **Local** environment of `client-research-agent` with one environment variable, so these build runs stay apart from the Development traffic Engine reads in Module 2. Model calls go through the LangSmith LLM Gateway with `LANGSMITH_GATEWAY_API_KEY`. It prints a link to this notebook's traces.

> On a project-based workspace (self-hosted or older orgs), the cell sends traces to a separate `<your project>-build` project instead.

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# utils.config loads the repo's .env, so import it before anything reads env vars
from utils.config import AGENT_NAME, BUILD_ENVIRONMENT, PROJECT_NAME, TRACING_MODE, environment_project
from utils.models import MODEL_ID, make_model
from utils.notebook import print_files, trace_link

from langgraph.checkpoint.memory import InMemorySaver
from langsmith import Client, uuid7

# Set before the first trace: keeps build runs apart from the Development traffic Engine reads in Module 2
if TRACING_MODE == "agent":
    os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = BUILD_ENVIRONMENT
else:
    os.environ["LANGSMITH_PROJECT"] = f"{PROJECT_NAME}-build"

model = make_model()
client = Client()

if TRACING_MODE == "agent":
    tracing_url = environment_project(client, BUILD_ENVIRONMENT).url
else:
    tracing_url = client.create_project(os.environ["LANGSMITH_PROJECT"], upsert=True).url


def trace_url(run_id):
    """Return the LangSmith URL of the trace whose root run is run_id."""
    return f"{tracing_url}/r/{run_id}?poll=true"


def tool_calls(result, name):
    """Return the main agent's calls to one tool, in order."""
    calls = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            if call["name"] == name:
                calls.append(call)
    return calls


print("Model:", MODEL_ID)
trace_link(tracing_url, "Open this notebook's traces")

## 1. The harness
`create_deep_agent()` is `create_agent()` plus a prebuilt middleware stack, so you get a filesystem, subagent delegation and context management out of the box, with no tools to write. The first cell builds the bare agent and lists its tools; the second asks it to write a haiku to a file and read it back.

**What you get for free:**

| Built-in | What it does |
|---|---|
| Filesystem tools | `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep` on a virtual filesystem; `execute` runs shell commands only with a sandbox backend |
| Subagent delegation | the `task` tool and a built-in `general-purpose` subagent for isolated work |
| Large tool result offloading | a result over ~20k tokens is saved to the filesystem, and the model gets its path and a preview |
| Conversation summarization | near ~85% of the model's context window, older history is summarized and the full messages are saved to the filesystem |
| Dangling tool-call patching | a tool call left without a result, for example after an interrupt, gets a placeholder so the history stays valid |
| Prompt caching | added automatically for Anthropic, Bedrock and Fireworks models |
| Planning (`write_todos`) | **not** in the default stack since deepagents 0.7; you add `TodoListMiddleware` in section 3 |

In [ ]:
from deepagents import create_deep_agent

# The checkpointer saves each thread's state, including its files, between calls
agent = create_deep_agent(model=model, checkpointer=InMemorySaver())

print("Built-in tools:", list(agent.nodes["tools"].bound.tools_by_name))

In [ ]:
HAIKU_REQUEST = "Write a haiku about markets to /haiku.txt, then read it back to me."
thread = {"thread_id": str(uuid7())}
run_id = uuid7()  # only so the cell can link to this trace

result = agent.invoke(
    {"messages": [{"role": "user", "content": HAIKU_REQUEST}]},
    config={"configurable": thread, "run_id": run_id},
)

print("Reply:", result["messages"][-1].text)
print("Files in state:", list(result["files"]))
trace_link(trace_url(run_id))

Open the trace: the model calls `write_file` and then `read_file`, and you wrote neither tool. The file lives in agent state, not on disk.

<img src="../images/TODO-01-local-environment.png" style="width: auto; max-height: 380px; border-radius: 8px;">

*The haiku trace under **Tracing** → `client-research-agent` → **Local**.*

The default **StateBackend** keeps files per thread through the checkpointer. The next cell asks for `/haiku.txt` again on the same thread.

In [ ]:
run_id = uuid7()
same_thread = agent.invoke(
    {"messages": [{"role": "user", "content": "Read /haiku.txt"}]},
    config={"configurable": thread, "run_id": run_id},
)

print("Same thread:", same_thread["messages"][-1].text)
trace_link(trace_url(run_id))

This cell lists the files from a new thread.

In [ ]:
run_id = uuid7()
new_thread = agent.invoke(
    {"messages": [{"role": "user", "content": "List all files with ls /"}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("New thread:", new_thread["messages"][-1].text)
trace_link(trace_url(run_id))

The same thread returns the haiku; the new thread finds no files. Section 5 makes some paths outlast the thread.

## 2. Your own tools and prompt
Tools you pass sit next to the built-in ones, and the `system_prompt` you pass is the agent's whole authored prompt. The first cell defines `web_search`, whose body calls `utils/search.py` (it retries Tavily and falls back to a canned result), and builds the agent; the second asks for a one-paragraph Apple summary written to `/summary.md`.

In [ ]:
from langchain_core.tools import tool
from utils.search import resilient_tavily_search


@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the web for company news, market data, or general information.

    Args:
        query: Search query to execute.
    """
    return resilient_tavily_search(query, max_retries=2)


SYSTEM_PROMPT = (
    "You are a client research assistant for a wealth management firm. "
    "Answer in plain language and cite the sources you used."
)

agent = create_deep_agent(model=model, tools=[web_search], system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())
print("Agent ready with web_search")

In [ ]:
SUMMARY_REQUEST = "Summarize recent news on Apple (AAPL) in one paragraph and write it to /summary.md. Search at most once."
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": SUMMARY_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("Reply:", result["messages"][-1].text)
print_files(result.get("files", {}))
trace_link(trace_url(run_id))

In the trace, `web_search` sits beside `write_file`. Open the first model run: its system prompt starts with `SYSTEM_PROMPT`.

## 3. Planning with to-dos
**TodoListMiddleware** adds a `write_todos` tool that keeps a task list in agent state, so a long task has a visible plan the model updates as it works. Since deepagents 0.7 you add it yourself; long multi-step tasks and smaller models benefit most. The first cell adds it to the agent; the second asks for an Apple-versus-Microsoft comparison and prints the final to-do list.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    middleware=[TodoListMiddleware()],
    checkpointer=InMemorySaver(),
)
print("Agent ready with write_todos")

In [ ]:
COMPARE_REQUEST = (
    "Plan your steps first, then compare the latest quarterly results of Apple and Microsoft "
    "and write the comparison to /compare.md. Search at most twice."
)
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": COMPARE_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("To-dos:")
for todo in result.get("todos", []):
    print(f"  {todo['status']:<12} {todo['content']}")
trace_link(trace_url(run_id))

Each `write_todos` call in the trace is a plan update: steps move from `pending` to `in_progress` to `completed`.

## 4. Subagents
A **subagent** runs in its own context: the main agent calls it with `task()` and sees only its final answer, so long research doesn't crowd the main context. The name and description tell the main agent when to use it, and a `general-purpose` subagent is always there unless you replace it. The first cell defines a `research-agent`; the second asks the main agent to delegate NVIDIA news research to it.

<img src="../images/deepAgentSubagents.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
from datetime import date

research_subagent = {
    "name": "research-agent",
    # The main agent reads this description to decide when to delegate
    "description": "Researches one company or market topic on the web. Give it one topic at a time.",
    "system_prompt": (
        f"You are a research assistant for a wealth management firm. Today is {date.today():%Y-%m-%d}. "
        "Use web_search at most 2 times, then answer with cited sources."
    ),
    "tools": [web_search],
}

agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    subagents=[research_subagent],
    checkpointer=InMemorySaver(),
)
print("Agent ready with subagent:", research_subagent["name"])

In [ ]:
NEWS_REQUEST = "Lightly research recent news on NVIDIA. Delegate the research to the research-agent."
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": NEWS_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("task calls:", len(tool_calls(result, "task")))
print("Reply:", result["messages"][-1].text[:1500])
trace_link(trace_url(run_id))

In the trace, the `task` run holds a full `research-agent` run with its own model calls and searches, while the main agent's context holds only its last message.

## 5. Memory across threads
**CompositeBackend** routes paths to different backends: here `/memories/` goes to a **StoreBackend**, which outlasts the thread, and everything else stays in state. You get long-term memory by choosing a path, with no memory tool to build. The first cell builds the agent; the second saves a client fact in one thread, and the third asks for it from a new thread.

<img src="../images/deepAgentBackends.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
backend = CompositeBackend(
    default=StateBackend(),
    # One shared namespace; per-user memory would put the user's ID here
    routes={"/memories/": StoreBackend(store=store, namespace=lambda runtime: ("memories", "shared"))},
)

MEMORY_PROMPT = (
    f"{SYSTEM_PROMPT} Save client facts under /memories/clients/. "
    "Check /memories/ before you answer a question about a client."
)
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=MEMORY_PROMPT,
    backend=backend,
    store=store,
    checkpointer=InMemorySaver(),
)
print("Agent ready with /memories/ in the store")

In [ ]:
REMEMBER_REQUEST = (
    "Remember that Smith Family Office's risk profile is aggressive growth. "
    "Save it to /memories/clients/smith-family-office.md"
)
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": REMEMBER_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("Thread 1:", result["messages"][-1].text)
print("Saved in the store:", [item.key for item in store.search(("memories", "shared"))])
trace_link(trace_url(run_id))

In [ ]:
RECALL_REQUEST = "What is Smith Family Office's risk profile? Check /memories/ first."
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": RECALL_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("Thread 2:", result["messages"][-1].text)
trace_link(trace_url(run_id))

Thread 2's trace starts with `ls` or `read_file` under `/memories/`: the answer comes from the store, not from the conversation.

## 6. Middleware
Middleware hooks into the agent loop: before and after the agent runs, before and after each model call, and around each model and tool call (`wrap_model_call`, `wrap_tool_call`). You add rules, logging or guardrails without touching the agent's tools or prompt. This section uses the two wrap hooks: the first cell defines a compliance-rules hook and an audit-trail hook, the second builds the agent with them, and the third asks for a Microsoft earnings summary.

<img src="../images/deepAgentMiddleware.png" style="width: auto; max-height: 380px; border-radius: 8px;">

**Built-in context management.** The harness itself is middleware, and three of its strategies keep a run inside the model's context window:

<img src="../images/Offloading Inputs LangChain.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Offload large inputs:** `write_file` and `edit_file` calls leave their full content in the history; as the context fills, older calls are truncated to a pointer to the file.

<img src="../images/Offloading Results LangChain.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Offload large results:** a tool result over ~20k tokens is written to the backend and replaced with its path and a preview; the agent can read or grep the full content later.

<img src="../images/LangChain Summarization.png" style="width: auto; max-height: 240px; border-radius: 8px;">

**Summarization:** near ~85% of the context window, older history is summarized, and the full messages are saved to the filesystem.

In [ ]:
from datetime import datetime

from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain_core.messages import SystemMessage

COMPLIANCE_RULES = (
    "## Compliance rules\n"
    "- Never include account numbers or passwords in a response.\n"
    "- Cite a source for every market fact.\n"
    "- Flag any request for non-public client information."
)
audit_log = []


@wrap_model_call
def compliance_rules(request, handler):
    # Appends to the system prompt on every model call; the agent's prompt stays unchanged
    blocks = list(request.system_message.content_blocks) if request.system_message else []
    blocks.append({"type": "text", "text": f"\n\n{COMPLIANCE_RULES}"})
    return handler(request.override(system_message=SystemMessage(content_blocks=blocks)))


@wrap_tool_call
def audit_trail(request, handler):
    result = handler(request)
    status = getattr(result, "status", "success")  # a Command result carries no status
    audit_log.append({"tool": request.tool_call["name"], "status": status, "at": datetime.now().strftime("%H:%M:%S")})
    return result


print("Middleware ready:", compliance_rules.name, audit_trail.name)

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    middleware=[compliance_rules, audit_trail],
    checkpointer=InMemorySaver(),
)
print("Agent ready with compliance_rules and audit_trail")

In [ ]:
EARNINGS_REQUEST = "Summarize Microsoft's latest earnings and write the summary to /summary.md. Search at most once."
audit_log.clear()
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": EARNINGS_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("Reply:", result["messages"][-1].text)
print("Audit log entries:", len(audit_log))
for entry in audit_log:
    print(f"  {entry['at']}  {entry['tool']:<20} {entry['status']}")
trace_link(trace_url(run_id))

The model run in the trace shows the compliance rules at the end of its system prompt, and the audit log has one row per tool call in the trace.

## 7. Human approval
`interrupt_on` is one application of middleware: it configures **HumanInTheLoopMiddleware**, which pauses the agent after the model proposes a call to a tool you list, so you approve, edit or reject it before it runs. You get an approval gate without writing pause-and-resume logic. The first cell gates `write_file` and `delete`; the second asks for a note on NVIDIA, which pauses at the write.

<img src="../images/deepAgentHITL.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    interrupt_on={"write_file": True, "delete": True},  # True allows approve, edit and reject
    checkpointer=InMemorySaver(),
)
print("Agent ready: write_file and delete need approval")

In [ ]:
NOTE_REQUEST = "Write a one-paragraph note on NVIDIA to /nvda_note.md. Search at most once."
config = {"configurable": {"thread_id": str(uuid7())}}
run_id = uuid7()

result = agent.invoke({"messages": [{"role": "user", "content": NOTE_REQUEST}]}, config={**config, "run_id": run_id})

for action in result["__interrupt__"][0].value["action_requests"]:
    print("Paused:", action["name"], action["args"].get("file_path"))
trace_link(trace_url(run_id))

The paused trace ends in a `GraphInterrupt` on `HumanInTheLoopMiddleware.after_model` with status interrupted: that is the pause, not a failure. The next cell approves each paused call and resumes the same thread from its checkpoint.

In [ ]:
from langgraph.types import Command

actions = result["__interrupt__"][0].value["action_requests"]
decisions = [{"type": "approve"} for _ in actions]  # one decision per paused call, in order
run_id = uuid7()

result = agent.invoke(Command(resume={"decisions": decisions}), config={**config, "run_id": run_id})

print("Approved:", len(decisions))
print("Reply:", result["messages"][-1].text)
print_files(result.get("files", {}))
trace_link(trace_url(run_id))

The resume is a second trace in the same thread, and it runs `write_file`. Open **Threads** in the Local environment to see both traces together.

## 8. AGENTS.md and skills
**Memory** files such as `AGENTS.md` load into the system prompt on every call, while **skills** load only their name and description up front, and the agent reads the full `SKILL.md` when a task matches. You change the agent's behavior by editing files, and skills keep long instructions out of the context until they're needed. Both are files the agent reads from its backend. The default backend keeps files in agent state, not on disk, so the cell below seeds them through the `files` key of the input.

In [ ]:
from deepagents.backends.utils import create_file_data

AGENTS_MD = """# Client Research Assistant

You research clients and markets for a wealth management firm and write pre-meeting briefs.

## Workflow
1. If a client is named, call get_client_profile first.
2. Delegate company and market research to the research-agent with task().
3. Write the brief to /client_brief.md.
4. Save key takeaways to /memories/client_notes.md.

## Rules
- Cite sources as [1], [2], [3] and end with a Sources list.
- Check /skills/ when asked for a specific format, such as a client brief.
"""

CLIENT_BRIEF_SKILL = """---
name: client-brief
description: Write a structured pre-meeting brief for a relationship manager. Use this skill when asked to prep a client meeting, summarize a client's situation, or produce a meeting prep document.
---

# Client Brief Skill

## Format
- **Header**: Client name, AUM, last meeting date
- **Portfolio Snapshot**: Top holdings with weights, risk profile (1-2 sentences)
- **Holdings Highlights**: 2-3 sentences per major holding covering recent news, key drivers, current risks
- **Market Context**: Brief paragraph on broader market themes relevant to the portfolio
- **Talking Points**: 3-5 bulleted prompts the RM can raise in the meeting
- **Sources**: Numbered list of citations referenced above

## Tone
- Concise and factual; lead with what changed since the last meeting; flag risks plainly.

## Length
- 400-600 words.
"""

seed_files = {
    "/AGENTS.md": create_file_data(AGENTS_MD),
    "/skills/client-brief/SKILL.md": create_file_data(CLIENT_BRIEF_SKILL),
}
print("Seed files:", list(seed_files))

The first cell builds an agent that loads `/AGENTS.md` and the skills under `/skills/`; the second asks for a client brief, which matches the `client-brief` skill's description.

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    subagents=[research_subagent],
    middleware=[TodoListMiddleware()],
    memory=["/AGENTS.md"],  # loaded into the system prompt on every call
    skills=["/skills/"],  # only each skill's name and description load up front
    checkpointer=InMemorySaver(),
)
print("Agent ready with AGENTS.md and skills")

In [ ]:
BRIEF_REQUEST = (
    "Prep a client brief for a family office that holds NVDA 12%, GOOGL 10% and BRK.B 8% "
    "and has an aggressive growth risk profile. Research one holding. Search at most once."
)
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": BRIEF_REQUEST}], "files": seed_files},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

print("Files read:", [call["args"].get("file_path") for call in tool_calls(result, "read_file")])
print("Reply:", result["messages"][-1].text[:1500])
trace_link(trace_url(run_id))

`Files read` lists `/skills/client-brief/SKILL.md`: the agent pulled the skill in only after the request matched its description. In the trace, that `read_file` call comes before the brief is written.

## 9. Interpreters (beta)
**CodeInterpreterMiddleware** gives the agent an `eval` tool that runs JavaScript in an in-memory QuickJS sandbox, so it can loop, branch and compute in code and return only the result to the model. Common uses are exact arithmetic and transforms over structured data (sort, group, score, aggregate), loops and retries across tools, and keeping large intermediate values out of the context. The sandbox has no network, filesystem or shell unless you bridge specific tools in.

The first cell builds an agent with the interpreter and a request with 8 holdings that tells it to compute the weighted one-year return and the top-3 concentration in JavaScript with `eval`. The second runs it and computes the same numbers in Python as a check.

> **Requires:** `langchain-quickjs` (installed with `deepagents[quickjs]` by `uv sync`) and Python 3.11+; interpreters are beta. **Without it:** skip to section 11; nothing later depends on sections 9 and 10.

In [ ]:
from langchain_quickjs import CodeInterpreterMiddleware

HOLDINGS = [
    ("NVDA", 0.12, 0.85), ("GOOGL", 0.10, 0.31), ("BRK.B", 0.08, 0.18), ("AAPL", 0.07, 0.22),
    ("MSFT", 0.07, 0.15), ("JPM", 0.05, 0.27), ("XOM", 0.04, -0.06), ("AGG", 0.47, 0.03),
]
holdings_table = "\n".join(f"{ticker}, {weight}, {one_year}" for ticker, weight, one_year in HOLDINGS)

# Without the last instruction the model often does the math itself and never calls eval
CODE_REQUEST = (
    "Compute this portfolio's weighted one-year return and the share held by its three largest positions. "
    "Do every calculation in JavaScript with the eval tool, not in your head.\n\n"
    f"ticker, weight, one_year_return\n{holdings_table}"
)

agent = create_deep_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    middleware=[CodeInterpreterMiddleware()],
    checkpointer=InMemorySaver(),
)
print("Agent ready with eval")

In [ ]:
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": CODE_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

expected_return = sum(weight * one_year for _, weight, one_year in HOLDINGS)
top_three = sum(sorted((weight for _, weight, _ in HOLDINGS), reverse=True)[:3])

print("Reply:", result["messages"][-1].text)
print("eval calls:", len(tool_calls(result, "eval")))
print(f"Python check: {expected_return:.2%} weighted return, {top_three:.0%} in the top three")
trace_link(trace_url(run_id))

The reply matches the Python check, and `eval calls` is at least 1. Open the `eval` run: its input is the JavaScript the model wrote and its output is the computed result, the only part that returns to the model.

<img src="../images/TODO-01-eval-run.png" style="width: auto; max-height: 380px; border-radius: 8px;">

*The `eval` run with the model's JavaScript as input.*

## 10. Dynamic subagents (beta)
With an interpreter and subagents together, `eval` code can call `task()` to dispatch subagents in loops or in parallel. That suits fan-out research, mapping one task over many items, and orchestration that runs deterministically in code instead of one model-chosen `task` call per turn. It is the "code mode" idea (Cloudflare's Code Mode, Anthropic's code execution with MCP: the model writes code that calls tools in a sandbox) applied to subagents, and the docs also use it for recursive language model (RLM) workflows, which keep a large input in interpreter variables and call a model on slices of it.

The first cell builds an agent with the interpreter and the `research-agent`; the second asks for a **workflow** (the documented trigger) that researches three tickers in parallel. The run takes about 1–2 minutes.

> **Requires:** the section 9 interpreter; dynamic subagents are beta. **Without it:** skip to section 11.

> `task()` calls from code skip `interrupt_on` approvals, so gate the `eval` tool itself if you need one.

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    subagents=[research_subagent],
    middleware=[CodeInterpreterMiddleware()],  # with subagents, eval code gets a task() function
    checkpointer=InMemorySaver(),
)
print("Agent ready with eval and research-agent")

In [ ]:
WORKFLOW_REQUEST = (
    "Run a workflow that researches NVDA, GOOGL and BRK.B in parallel with the research-agent, "
    "one task per ticker, then list the top risk for each. One search per ticker."
)
run_id = uuid7()

result = agent.invoke(
    {"messages": [{"role": "user", "content": WORKFLOW_REQUEST}]},
    config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
)

eval_code = [call["args"]["code"] for call in tool_calls(result, "eval")]
print("Reply:", result["messages"][-1].text)
print("eval calls:", len(eval_code))
print("task() in code:", any("task(" in code for code in eval_code))
trace_link(trace_url(run_id))

The trace has one `eval` run with three `research-agent` runs inside it that start at the same time; in section 4 the model made one `task` call per step instead.

<img src="../images/TODO-01-parallel-subagents.png" style="width: auto; max-height: 380px; border-radius: 8px;">

*Three subagent runs nested under one `eval` run.*

## 11. The complete agent
This agent combines every piece and adds the real client lookup, `get_client_profile`, from `agents/client_research_agent.py`, the tool Modules 2–5 use. `client_lookup_errors`, the same `ToolErrorMiddleware` the packaged agent uses, hands lookup errors back to the model so the run keeps going. The first cell builds it; the second starts a meeting-prep request, which pauses at the first file write; the third approves each write until the agent finishes. The run takes about 2–3 minutes and makes several traces in one thread: the first request, then one per resume.

In [ ]:
from agents.client_research_agent import client_lookup_errors, get_client_profile

store = InMemoryStore()
complete_agent = create_deep_agent(
    model=model,
    tools=[web_search, get_client_profile],
    subagents=[research_subagent],
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(store=store, namespace=lambda runtime: ("memories", "shared"))},
    ),
    store=store,
    middleware=[
        TodoListMiddleware(),
        compliance_rules,
        audit_trail,
        client_lookup_errors,  # returns lookup errors to the model instead of raising
    ],
    interrupt_on={"write_file": True, "delete": True},
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
)
print("Complete agent: tools, subagent, memory, planning, middleware, approvals, AGENTS.md, skills")

In [ ]:
MEETING_REQUEST = "Prep me for my meeting with Smith Family Office. Follow your AGENTS.md workflow. Search at most once."
config = {"configurable": {"thread_id": str(uuid7())}}
audit_log.clear()
run_id = uuid7()

result = complete_agent.invoke(
    {"messages": [{"role": "user", "content": MEETING_REQUEST}], "files": seed_files},
    config={**config, "run_id": run_id},
)

print("Paused:", bool(result.get("__interrupt__")))
trace_link(trace_url(run_id), "View the first trace")

In [ ]:
resumes = 0
while result.get("__interrupt__"):
    actions = result["__interrupt__"][0].value["action_requests"]
    for action in actions:
        print("Approving:", action["name"], action["args"].get("file_path"))

    decisions = [{"type": "approve"} for _ in actions]
    result = complete_agent.invoke(Command(resume={"decisions": decisions}), config=config)
    resumes += 1

# Seed files are inputs; show only what the agent wrote
agent_files = {path: data for path, data in result.get("files", {}).items() if path not in seed_files}

print("Resumes:", resumes)
print("Reply:", result["messages"][-1].text[:600])
print_files(agent_files)
print("Audit log entries:", len(audit_log))
for entry in audit_log:
    print(f"  {entry['at']}  {entry['tool']:<20} {entry['status']}")

Open the first trace and count the `get_client_profile` calls before the brief, then compare them with the `error` rows in the audit log. **Threads** in the Local environment shows that trace and each resume together. Module 2 comes back to the lookup errors.

## Recap
| Step | Where it lives |
|---|---|
| Harness | `create_deep_agent()` |
| Your tools and prompt | `tools=`, `system_prompt=` |
| Planning | `TodoListMiddleware` → `write_todos` |
| Subagents | `subagents=`, the `task` tool |
| Memory across threads | `CompositeBackend` + `StoreBackend(namespace=...)` |
| Middleware | `@wrap_model_call`, `@wrap_tool_call` |
| Approvals | `interrupt_on=`, resume with `Command(resume=...)` |
| AGENTS.md and skills | `memory=["/AGENTS.md"]`, `skills=["/skills/"]` |
| Interpreters (beta) | `CodeInterpreterMiddleware` → `eval` |
| Dynamic subagents (beta) | `task()` inside `eval` |
| Traces | **Tracing** → `client-research-agent` → **Local** |

`agents/client_research_agent.py` packages a slimmer version of this agent, `build_client_research_agent()`, which Modules 2–5 run.

**Next:** Module 2 — Tracing

**Docs:** [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) · [Backends](https://docs.langchain.com/oss/python/deepagents/backends) · [Context engineering](https://docs.langchain.com/oss/python/deepagents/context-engineering) · [Human-in-the-loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop) · [Skills](https://docs.langchain.com/oss/python/deepagents/skills) · [Interpreters](https://docs.langchain.com/oss/python/deepagents/interpreters) · [Dynamic subagents](https://docs.langchain.com/oss/python/deepagents/dynamic-subagents) · [Log traces to an agent](https://docs.langchain.com/langsmith/log-traces-to-agent)